In [ ]:
import numpy as np
import nibabel as nib
from nilearn.image import resample_img
from util import collapse_array_3d, collapse_slices_3d


In [30]:

# To test resampling, create a fake, very small, image.
# Then resample it with different methods
# and review the resulting lower-res image.
# This will be 8 x 8 x 6, with 2.0mm voxels,
# then downsampled to 4 x 4 x 3, with 4.0mm voxels.
a = np.zeros((10, 10, 7))
# Manually draw a blurred sphere.
# The center of the sphere is between voxels at (3.5, 3.5, 2.5).
# We can assign a higher value by nearness to the center.
center = (4.5, 4.5, 3)
max_value = 100.0
for x in range(10):
    for y in range(10):
        for z in range(7):
            dist_from_center = np.sqrt(np.sum([
                (center[0] - x)**2,
                (center[1] - y)**2,
                (center[2] - z)**2,
            ]))
            if dist_from_center == 0.0:
                a[x, y, z] = max_value
            else:
                a[x, y, z] = (max_value / 2) / dist_from_center

# Tweak a single voxel to see if it's averaged into the lo-res or sampled.
a[2, 5, 2] = 0.125

print(np.unique(a.ravel(), return_counts=True))


(array([ 0.125     ,  7.10669055,  7.49531689,  7.76150526,  7.85674201,
        8.27605889,  8.39181358,  8.63868426,  8.77058019,  8.90870806,
        9.0535746 ,  9.20574618,  9.36585812,  9.53462589,  9.71285862,
        9.90147543, 10.10152545, 10.31421246, 10.54092553, 10.78327732,
       11.04315261, 11.32277034, 11.62476387, 11.95228609, 12.3091491 ,
       12.7000127 , 13.13064329, 13.60827635, 14.14213562, 14.74419562,
       15.430335  , 16.22214211, 17.14985851, 18.25741858, 19.61161351,
       21.32007164, 23.57022604, 26.72612419, 31.6227766 , 40.82482905,
       70.71067812]), array([ 1,  8,  8, 24,  4, 16, 16, 24,  8, 16, 16, 16,  8, 32, 24,  8, 20,
       32, 24, 40,  8, 16, 24, 16, 24, 32,  8, 32, 28, 16, 16, 24, 16, 15,
       24,  8, 12, 16,  8,  8,  4]))


In [31]:

# Make the data represent 2.0mm voxels in 2.5mm slices,
# and flip the x for radiologic orientation, just to make it more
# interesting
orig_affine = np.array([
    [-2.0, 0.0, 0.0, -6.0, ],
    [0.0, 2.0, 0.0, -2.0, ],
    [0.0, 0.0, 2.5, 0.0, ],
    [0.0, 0.0, 0.0, 1.0, ],
])
orig_img = nib.Nifti2Image(a, orig_affine)
orig_img.to_filename("/mnt/cache/stare_work/resamp/fake_hi.nii.gz")
print(orig_img.affine)
print(orig_img.shape)

[[-2.   0.   0.  -6. ]
 [ 0.   2.   0.  -2. ]
 [ 0.   0.   2.5  0. ]
 [ 0.   0.   0.   1. ]]
(10, 10, 7)


In [33]:

samp_affine = orig_affine.copy()
samp_affine[:3, :3] = orig_affine[:3, :3] * 2.0
samp_img_c = resample_img(
    orig_img,
    target_affine=samp_affine, target_shape=(5, 5, 4),
    interpolation='continuous', copy_header=True,
    force_resample=True,
)
samp_img_c.to_filename("/mnt/cache/stare_work/resamp/fake_lo_c.nii.gz")
print(samp_affine)
print(samp_img_c.affine)
print(samp_img_c.shape)


[[-4.  0.  0. -6.]
 [ 0.  4.  0. -2.]
 [ 0.  0.  5.  0.]
 [ 0.  0.  0.  1.]]
[[-4.  0.  0. -6.]
 [ 0.  4.  0. -2.]
 [ 0.  0.  5.  0.]
 [ 0.  0.  0.  1.]]
(5, 5, 4)


In [39]:
samp_img_l = resample_img(
    orig_img,
    target_affine=samp_affine, target_shape=(5, 5, 4),
    interpolation='linear', copy_header=True,
    force_resample=True,
)
samp_img_l.to_filename("/mnt/cache/stare_work/resamp/fake_lo_l.nii.gz")
print(samp_img_l.affine)
print(samp_img_l.shape)


[[-4.  0.  0. -6.]
 [ 0.  4.  0. -2.]
 [ 0.  0.  5.  0.]
 [ 0.  0.  0.  1.]]
(5, 5, 4)


In [40]:
samp_img_n = resample_img(
    orig_img,
    target_affine=samp_affine, target_shape=(5, 5, 4),
    interpolation='nearest', copy_header=True,
    force_resample=True,
)
samp_img_n.to_filename("/mnt/cache/stare_work/resamp/fake_lo_n.nii.gz")
print(samp_img_n.affine)
print(samp_img_n.shape)


[[-4.  0.  0. -6.]
 [ 0.  4.  0. -2.]
 [ 0.  0.  5.  0.]
 [ 0.  0.  0.  1.]]
(5, 5, 4)


In [59]:
[slice(None)] * 2 + [np.s_[6:8]]

[slice(None, None, None), slice(None, None, None), slice(6, 8, None)]

In [67]:
orig_img.get_fdata().ndim


3

In [ ]:
custom_data = np.zeros(orig_img.shape)
print(custom_data.shape)

In [86]:
new_array = collapse_array_3d(custom_data, by=2)
print(new_array)


reshaping data from (10, 10, 7) to (10, 10, 4)
  0: (10, 10)
  1: (10, 10)
  2: (10, 10)
  3: (10, 10)
reshaping data from (10, 10, 4) to (10, 5, 4)
  0: (10, 4)
  1: (10, 4)
  2: (10, 4)
  3: (10, 4)
  4: (10, 4)
reshaping data from (10, 5, 4) to (5, 5, 4)
  0: (5, 4)
  1: (5, 4)
  2: (5, 4)
  3: (5, 4)
  4: (5, 4)
[[[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]]


In [81]:
my_along = 1
new_dim = int(np.ceil(custom_data.shape[my_along] / 2.0))
print(new_dim)
print(*custom_data.shape[:my_along], new_dim, *custom_data.shape[my_along + 1:])

collapse_slices_3d(custom_data, by=2, along=my_along)


5
10 5 7
reshaping data from (10, 10, 7) to (10, 5, 7)
  0: (10, 7)
  1: (10, 7)
  2: (10, 7)
  3: (10, 7)
  4: (10, 7)


array([[[0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0.],
  

In [ ]:

for z in range(4):
    # set range of z values to average
    z0 = z * 2
    z1 = z * 2 + 2
    if z1 > 4:
        z1 = 4
    new_slice = np.mean(orig_img.get_fdata()[:, :, z0:z1], axis=2)
    print(f"{z}: {new_slice.shape}")
    custom_data[:, :, z] = new_slice
print(custom_data.shape)
for y in range(5):
    # set range of y values to average
    y0 = y * 2
    y1 = y * 2 + 2
    if y1 > 5:
        y1 = 5
    new_slice = np.mean(orig_img.get_fdata()[:, y0:y1, :], axis=1)
    print(f"{y}: {new_slice.shape}")
    custom_data[:, y, :] = new_slice
print(custom_data.shape)
for x in range(5):
    # set range of y values to average
    x0 = x * 2
    x1 = x * 2 + 2
    if x1 > 5:
        x1 = 5
    new_slice = np.mean(orig_img.get_fdata()[x0:x1, :, :], axis=0)
    print(f"{z}: {new_slice.shape}")
    custom_data[x, :, :] = new_slice
print(custom_data.shape)

In [87]:

samp_img_custom = nib.Nifti2Image(
    collapse_array_3d(orig_img.get_fdata()),
    samp_affine
)

reshaping data from (10, 10, 7) to (10, 10, 4)
  0: (10, 10)
  1: (10, 10)
  2: (10, 10)
  3: (10, 10)
reshaping data from (10, 10, 4) to (10, 5, 4)
  0: (10, 4)
  1: (10, 4)
  2: (10, 4)
  3: (10, 4)
  4: (10, 4)
reshaping data from (10, 5, 4) to (5, 5, 4)
  0: (5, 4)
  1: (5, 4)
  2: (5, 4)
  3: (5, 4)
  4: (5, 4)


In [43]:
ax_0_shape = np.mean(orig_img.get_fdata(), axis=0).shape
ax_1_shape = np.mean(orig_img.get_fdata(), axis=1).shape
ax_2_shape = np.mean(orig_img.get_fdata(), axis=2).shape
print(ax_0_shape, ax_1_shape, ax_2_shape)


(10, 7) (10, 7) (10, 10)


In [34]:
print(np.unique(samp_img_c.get_fdata().ravel(), return_counts=True))

(array([ 7.10669055,  7.10669055,  7.76150526,  7.76150526,  7.76150526,
        7.76150526,  7.76150526,  8.39181358,  8.39181358,  8.39181358,
        8.39181358,  8.63868426,  8.63868426,  8.90870806,  8.90870806,
        8.90870806,  8.90870806,  9.20574618,  9.20574618,  9.20574618,
        9.53462589,  9.53462589,  9.53462589,  9.53462589,  9.90147543,
        9.90147543, 10.31421246, 10.31421246, 10.31421246, 10.31421246,
       10.31421246, 10.31421246, 10.31421246, 10.78327732, 10.78327732,
       10.78327732, 10.78327732, 11.32277034, 11.32277034, 11.32277034,
       11.95228609, 11.95228609, 11.95228609, 12.7000127 , 12.7000127 ,
       12.7000127 , 12.7000127 , 13.60827635, 13.60827635, 13.60827635,
       13.60827635, 13.60827635, 14.74419562, 14.74419562, 14.74419562,
       16.22214211, 16.22214211, 16.22214211, 18.25741858, 21.32007164,
       21.32007164, 26.72612419, 40.82482905, 40.82482905]), array([1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 1, 1, 1, 1, 2, 1, 2,
 

In [35]:
np.mean(orig_img.get_fdata()[:2, :2, :2])


np.float64(8.085209761618774)

In [36]:
orig_img.get_fdata()[:2, :2, :2]


array([[[7.10669055, 7.49531689],
        [7.76150526, 8.27605889]],

       [[7.76150526, 8.27605889],
        [8.63868426, 9.36585812]]])

In [88]:
for desc, mat in [
    ("continuous", samp_img_c),
    ("linear", samp_img_l),
    ("nearest", samp_img_n),
    ("custom", samp_img_custom),
]:
    print(f"{desc}: (0, 0, 0)={mat.get_fdata()[0, 0, 0]:0.5f}, "
          f"(1, 2, 1)={mat.get_fdata()[1, 2, 1]:0.5f}",
          f"(1, 3, 1)={mat.get_fdata()[1, 3, 1]:0.5f}")


continuous: (0, 0, 0)=7.10669, (1, 2, 1)=18.25742 (1, 3, 1)=16.22214
linear: (0, 0, 0)=7.10669, (1, 2, 1)=18.25742 (1, 3, 1)=16.22214
nearest: (0, 0, 0)=7.10669, (1, 2, 1)=18.25742 (1, 3, 1)=16.22214
custom: (0, 0, 0)=8.08521, (1, 2, 1)=21.78793 (1, 3, 1)=17.42309


In [ ]:
samp_img_l.get_fdata()[0, 0, 0]
